# Materialized Lake Views - Data Quality

**Companion to `mlv_get_started`**

The previous notebook is assumed to have run successfully. Attach the same
**`mlv_lakehouse`** as this notebook's default Lakehouse.

This short demo creates five MLVs over **`bronze.orders`**, inserts two invalid orders,
and compares the outcomes. No extra source tables are created.

**Run cells individually:** three refreshes intentionally fail. After each expected
data-quality error, continue with the next cell; **Run all** will stop.

The demo writes to the original `bronze.orders` table. Use it in your tutorial Lakehouse,
reserve order IDs **990001 and 990002**, and run the optional cleanup before repeating it.
Existing MLV definitions stay unchanged, but their next scheduled refresh can pick up
these orders. Avoid concurrent scheduled refreshes during the demo.


## Step 1: Create One MLV per Outcome

Each checked view requires a positive quantity and a non-null order date.
Explicit null checks make the contract unambiguous.

| View | Behavior on invalid data |
|---|---|
| `silver.dq_orders_unchecked` | Keeps every row |
| `silver.dq_orders_drop` | Drops invalid rows; refresh succeeds |
| `silver.dq_orders_fail` | Fails the refresh |
| `silver.dq_orders_default` | Also fails: FAIL is the default |
| `silver.dq_orders_mixed` | Quantity uses DROP, date uses FAIL; FAIL takes precedence |

`CREATE OR REPLACE` replaces only these demo MLVs, not the original tutorial views.


In [1]:
%%sql
-- Control branch: no data-quality contract.
CREATE OR REPLACE MATERIALIZED LAKE VIEW silver.dq_orders_unchecked AS
SELECT * FROM bronze.orders;

StatementMeta(, a0d06b14-5918-48a5-8f01-5722f5d0c3d6, 2, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 2 fields>

In [ ]:
%%sql
-- DROP removes invalid rows from the MLV, not from bronze.orders.
CREATE OR REPLACE MATERIALIZED LAKE VIEW silver.dq_orders_drop
(
  CONSTRAINT positive_quantity CHECK (quantity IS NOT NULL AND quantity > 0) ON MISMATCH DROP,
  CONSTRAINT required_date CHECK (order_date IS NOT NULL) ON MISMATCH DROP
)
AS SELECT * FROM bronze.orders;

In [ ]:
%%sql
-- FAIL rejects the refresh when either part of the contract is violated.
CREATE OR REPLACE MATERIALIZED LAKE VIEW silver.dq_orders_fail
(
  CONSTRAINT valid_order CHECK (
    quantity IS NOT NULL AND quantity > 0 AND order_date IS NOT NULL
  ) ON MISMATCH FAIL
)
AS SELECT * FROM bronze.orders;

In [ ]:
%%sql
-- Omitting ON MISMATCH does not disable the check: the default is FAIL.
CREATE OR REPLACE MATERIALIZED LAKE VIEW silver.dq_orders_default
(
  CONSTRAINT valid_order CHECK (
    quantity IS NOT NULL AND quantity > 0 AND order_date IS NOT NULL
  )
)
AS SELECT * FROM bronze.orders;

In [ ]:
%%sql
-- A DROP rule does not hide a violation of a FAIL rule.
CREATE OR REPLACE MATERIALIZED LAKE VIEW silver.dq_orders_mixed
(
  CONSTRAINT positive_quantity CHECK (quantity IS NOT NULL AND quantity > 0) ON MISMATCH DROP,
  CONSTRAINT required_date CHECK (order_date IS NOT NULL) ON MISMATCH FAIL
)
AS SELECT * FROM bronze.orders;

## Step 2: Check the Clean Baseline

Refresh and compare counts. **All six counts should match** because the original orders
are valid: normally **3 rows** after `mlv_get_started`, or **N** if you added more valid orders.

FULL refreshes are used here only for a predictable, one-time demo.
Use Lakehouse lineage and schedules for recurring refresh.


In [ ]:
%%sql
-- Materialize the same clean input in every branch.
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_unchecked FULL;
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_drop FULL;
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_fail FULL;
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_default FULL;
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_mixed FULL;

SELECT 'source' AS branch, COUNT(*) AS row_count FROM bronze.orders
UNION ALL SELECT 'unchecked', COUNT(*) FROM silver.dq_orders_unchecked
UNION ALL SELECT 'drop', COUNT(*) FROM silver.dq_orders_drop
UNION ALL SELECT 'fail', COUNT(*) FROM silver.dq_orders_fail
UNION ALL SELECT 'default', COUNT(*) FROM silver.dq_orders_default
UNION ALL SELECT 'mixed', COUNT(*) FROM silver.dq_orders_mixed;

In [ ]:
%%sql
-- All original orders should be present; no rows have been dropped.
SELECT * FROM silver.dq_orders_drop ORDER BY order_id;


## Step 3: Insert Two Invalid Orders

Insert directly into the **original source table**, not into the MLVs.

| Order | Quantity | Date | Violations |
|---|---|---|---|
| 990001 | 0 | 2025-06-10 | Quantity |
| 990002 | -1 | NULL | Quantity and date |

The second row tests mixed actions: even though its quantity is droppable,
its missing date must still cause FAIL. Execute this insert **once per demo**.


In [ ]:
%%sql
-- Both orders reference an existing product from the base tutorial.
INSERT INTO bronze.orders VALUES
  (990001, 101,  0, DATE '2025-06-10'),
  (990002, 101, -1, NULL);

In [ ]:
%%sql
SELECT * FROM bronze.orders
ORDER BY order_id;

## Step 4: Refresh and Compare the Outcomes

First refresh the unchecked and DROP branches. Both succeed, but only the unchecked
view contains the two invalid orders. **DROP does not repair or delete source rows.**


In [ ]:
%%sql
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_unchecked FULL;
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_drop FULL;

-- Expected: two rows under "unchecked", no rows under "drop".
SELECT 'unchecked' AS branch, * FROM silver.dq_orders_unchecked
WHERE order_id IN (990001, 990002)
UNION ALL
SELECT 'drop', * FROM silver.dq_orders_drop
WHERE order_id IN (990001, 990002)
ORDER BY branch, order_id;


### Explicit FAIL - Expected Error

Run the next cell and inspect the constraint violation. The refresh should fail,
not silently discard the invalid orders. Then continue manually.


In [ ]:
%%sql
-- EXPECTED DATA-QUALITY ERROR.
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_fail FULL;


### Default Action - Expected Error

The same result without an explicit action: **the default is FAIL**.
Run the cell, inspect the error, and continue manually.


In [ ]:
%%sql
-- EXPECTED DATA-QUALITY ERROR: ON MISMATCH was omitted.
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_default FULL;


### Mixed DROP and FAIL - Expected Error

Order 990002 violates both rules. **FAIL takes precedence**; do not assume DROP runs
first and hides the missing date. Run the cell, then continue manually.


In [ ]:
%%sql
-- EXPECTED DATA-QUALITY ERROR: required_date uses FAIL.
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_mixed FULL;


### Final Comparison

| Branch | Clean baseline | After inserting two invalid orders |
|---|---|---|
| Source / unchecked | N | N+2 |
| DROP | N | N; invalid rows excluded |
| FAIL / default / mixed | N | Refresh fails; previous N rows remain |

With the original three orders, expect **5, 5, 3, 3, 3, 3** below.
A failed refresh can leave old data readable: **available does not mean up to date**.
Check refresh errors as well as query results.

For managed runs, inspect the Lakehouse **Materialized lake views > Manage** lineage
and run details to see failures and DROP counts.


In [ ]:
%%sql
-- Compare with the baseline counts from Step 2.
SELECT 'source' AS branch, COUNT(*) AS row_count FROM bronze.orders
UNION ALL SELECT 'unchecked', COUNT(*) FROM silver.dq_orders_unchecked
UNION ALL SELECT 'drop', COUNT(*) FROM silver.dq_orders_drop
UNION ALL SELECT 'fail', COUNT(*) FROM silver.dq_orders_fail
UNION ALL SELECT 'default', COUNT(*) FROM silver.dq_orders_default
UNION ALL SELECT 'mixed', COUNT(*) FROM silver.dq_orders_mixed;


In [ ]:
%%sql
-- Compare actual contents: only "source" and "unchecked" should return rows.
SELECT 'source' AS branch, * FROM bronze.orders
WHERE order_id IN (990001, 990002)
UNION ALL
SELECT 'unchecked', * FROM silver.dq_orders_unchecked
WHERE order_id IN (990001, 990002)
UNION ALL
SELECT 'drop', * FROM silver.dq_orders_drop
WHERE order_id IN (990001, 990002)
UNION ALL
SELECT 'fail', * FROM silver.dq_orders_fail
WHERE order_id IN (990001, 990002)
UNION ALL
SELECT 'default', * FROM silver.dq_orders_default
WHERE order_id IN (990001, 990002)
UNION ALL
SELECT 'mixed', * FROM silver.dq_orders_mixed
WHERE order_id IN (990001, 990002)
ORDER BY branch, order_id;


## Optional: PySpark MLV Example (Preview)

The same DROP contract can be expressed with **`fmlv` decorators**.
Run this example while the two invalid orders are still in the source.

PySpark-authored MLVs use **full refresh**, with subsequent refresh managed through
Lakehouse lineage/schedules rather than SQL notebook refresh. `replace=True` replaces
the definition; do not assume it immediately updates an existing materialization.

The explicit DataFrame filter also makes the returned data clean during initial
materialization; the decorators register the durable refresh-time checks.
After a managed refresh, querying the persisted MLV should return **N valid orders**.


In [ ]:
import fmlv
from pyspark.sql import functions as F


@fmlv.materialized_lake_view(name="silver.dq_orders_pyspark", replace=True)
@fmlv.check(
    name="positive_quantity",
    condition="quantity IS NOT NULL AND quantity > 0",
    action="DROP",
)
@fmlv.check(
    name="required_date",
    condition="order_date IS NOT NULL",
    action="DROP",
)
def dq_orders_pyspark():
    # Read the same original table used by the Spark SQL examples.
    return spark.table("bronze.orders").filter(
        F.col("quantity").isNotNull()
        & (F.col("quantity") > 0)
        & F.col("order_date").isNotNull()
    )


After refreshing this optional MLV through Lakehouse lineage, inspect its stored rows.
The two demo orders should be absent.

**Note:** this is a row-level DROP example, not a column-level DQ suite.
SQL `CHECK` supports DROP and FAIL; WARN belongs to the separate column-level DQ API.


In [ ]:
%%sql
-- Run after the optional PySpark MLV has been materialized.
SELECT * FROM silver.dq_orders_pyspark ORDER BY order_id;


## Optional: Restore the Source for the Next Demo

Remove only the two reserved demo orders, then refresh the five SQL branches.
Every refresh should succeed again. The original tutorial MLVs are not refreshed here;
their next managed refresh will reflect the restored source.
If you created the optional PySpark MLV, refresh it through lineage as well.


In [ ]:
%%sql
-- Optional cleanup: these IDs must be reserved exclusively for this demo.
DELETE FROM bronze.orders WHERE order_id IN (990001, 990002);

REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_unchecked FULL;
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_drop FULL;
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_fail FULL;
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_default FULL;
REFRESH MATERIALIZED LAKE VIEW silver.dq_orders_mixed FULL;

SELECT * FROM silver.dq_orders_fail ORDER BY order_id;


### Learn More

- [Data quality in MLVs](https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/data-quality)
- [Spark SQL reference](https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/create-materialized-lake-view)
- [PySpark MLVs (Preview)](https://learn.microsoft.com/en-us/fabric/data-engineering/materialized-lake-views/create-materialized-lake-view-pyspark)
